In [7]:
from pathlib import Path
import json
import pandas as pd

In [9]:
project_dir = Path.cwd()
data_dir = project_dir / "data"

print("Project folder:", project_dir)
print("Data folder exists:", data_dir.exists())

Project folder: C:\Users\sonal\OneDrive\Desktop\case_study_jp
Data folder exists: True


In [11]:
for file in sorted(data_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(data_dir))

atlas\atlas_usage_20260817.xlsx
atlas\atlas_usage_20260824.xlsx
atlas\atlas_usage_20260831.xlsx
atlas\atlas_usage_20260907.xlsx
atlas\atlas_usage_20260913.xlsx
atlas\atlas_usage_20260914.xlsx
atlas\atlas_usage_20260921.xlsx
atlas\atlas_usage_20261005.xlsx
hr_roster.csv
quill_events.csv
reporting_config.json


In [13]:
with open(data_dir / "reporting_config.json", encoding="utf-8") as file:
    config = json.load(file)

print(json.dumps(config, indent=4))    # json string

{
    "reporting_window": {
        "first_week_start": "2026-08-09",
        "last_week_start": "2026-09-27"
    },
    "week_definition": "Weeks run Sunday 00:00 UTC to Saturday 23:59:59 UTC. week_start is the Sunday date.",
    "scope": {
        "division": "ADV",
        "division_name": "Advisory",
        "hr_rule": "Use each person's most recent HR record (latest Effective_Date) for every week."
    },
    "tools": [
        {
            "tool_id": "quill",
            "source": "quill_events.csv",
            "wau_metric": "count of event_type == 'generation'",
            "wau_threshold": 1,
            "wpu_threshold": 12
        },
        {
            "tool_id": "atlas",
            "source": "atlas/atlas_usage_<YYYYMMDD extract date>.xlsx",
            "wau_metric": "Active Days",
            "wau_threshold": 1,
            "wpu_metric": "Queries Run",
            "wpu_threshold": 25
        }
    ]
}


In [15]:
print("First week:", config["reporting_window"]["first_week_start"])
print("Last week:", config["reporting_window"]["last_week_start"])
print("Required division:", config["scope"]["division"])

First week: 2026-08-09
Last week: 2026-09-27
Required division: ADV


**loading Data - Hr Roster**

**Data Inspection**

In [23]:
hr = pd.read_csv(data_dir / "hr_roster.csv")    #loading Hr Roter data set

hr.head()

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
0,E8456729,ADV,People Advisory,AMER,L5,2023-11-13
1,E3961175,ADV,Strategy,APAC,L1,2024-09-19
2,E2473900,AUD,Audit Assurance,AMER,L5,2023-02-22
3,E9720776,TAX,Corporate Tax,AMER,L4,2026-04-11
4,E5492817,SHS,IT Services,AMER,L2,2023-01-16


In [ ]:
print("Rows and columns:", hr.shape)
print("Unique employees:", hr["Employee_ID"].nunique())

hr.info()

In [30]:
print("Rows and columns:", hr.shape)                         # how many rows & columns
print("Unique employees:", hr["Employee_ID"].nunique())

hr.info()

Rows and columns: (2445, 6)
Unique employees: 2400
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2445 entries, 0 to 2444
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     2445 non-null   object
 1   Division        2445 non-null   object
 2   Department      2445 non-null   object
 3   Region          2445 non-null   object
 4   Job_Level       2445 non-null   object
 5   Effective_Date  2445 non-null   object
dtypes: object(6)
memory usage: 114.7+ KB


In [32]:
print("Missing values by column:")
print(hr.isna().sum())

print("\nExact duplicate rows:", hr.duplicated().sum())              # every column matches another row

Missing values by column:
Employee_ID       0
Division          0
Department        0
Region            0
Job_Level         0
Effective_Date    0
dtype: int64

Exact duplicate rows: 0


In [38]:
multiple_records = hr[hr.duplicated(subset="Employee_ID", keep=False)]   # keep=False = marks all occurrences of a repeated employee ID

multiple_records.sort_values(["Employee_ID", "Effective_Date"]).head(10)

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
1730,E0162274,SHS,Finance Operations,EMEA,L3,2026-01-08
2053,E0162274,ADV,Risk,EMEA,L3,2026-07-12
944,E0171370,ADV,Technology Consulting,AMER,L1,2023-08-14
369,E0171370,ADV,People Advisory,AMER,L1,2026-09-01
1272,E0588098,ADV,Strategy,AMER,L1,2025-09-14
1208,E0588098,ADV,Deals,AMER,L1,2026-09-01
33,E0668929,ADV,Deals,AMER,L4,2025-12-29
924,E0668929,SHS,IT Services,AMER,L4,2026-08-14
492,E0907126,AUD,Audit Assurance,AMER,L4,2024-08-26
1451,E0907126,ADV,Deals,AMER,L4,2026-08-19


## HR data preparation
Standardise employee IDs, select each employee's latest HR record,
and retain employees in the Advisory division.

In [41]:
# Keep the original HR data unchanged for comparison.

hr_clean = hr.copy()

In [43]:
# Remove surrounding spaces and convert IDs to uppercase.

hr_clean["Employee_ID"] = hr_clean["Employee_ID"].str.strip().str.upper()


hr_clean.head()

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
0,E8456729,ADV,People Advisory,AMER,L5,2023-11-13
1,E3961175,ADV,Strategy,APAC,L1,2024-09-19
2,E2473900,AUD,Audit Assurance,AMER,L5,2023-02-22
3,E9720776,TAX,Corporate Tax,AMER,L4,2026-04-11
4,E5492817,SHS,IT Services,AMER,L2,2023-01-16


In [45]:
# Convert text dates into actual dates so we can order records correctly.
# The source format is year-month-day, for example 2026-08-14.

hr_clean["Effective_Date"] = pd.to_datetime(hr_clean["Effective_Date"],format="%Y-%m-%d")

# Check that Effective_Date now has a datetime type.
hr_clean.dtypes

Employee_ID               object
Division                  object
Department                object
Region                    object
Job_Level                 object
Effective_Date    datetime64[ns]
dtype: object

In [47]:
# Group each employee's records together and order them by date.

hr_sorted = hr_clean.sort_values(by=["Employee_ID", "Effective_Date"])

hr_sorted.head(10)

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
1831,E0001338,AUD,Audit Assurance,EMEA,L1,2026-01-07
2150,E0005667,ADV,Technology Consulting,APAC,L3,2025-05-07
2334,E0006083,ADV,People Advisory,APAC,L3,2024-03-20
2064,E0006249,ADV,People Advisory,EMEA,L2,2024-07-02
2058,E0014152,SHS,Finance Operations,AMER,L3,2026-04-08
1147,E0017798,ADV,Risk,APAC,L1,2024-03-12
421,E0024994,ADV,Deals,AMER,L1,2024-01-07
1984,E0028755,ADV,Strategy,AMER,L4,2024-04-10
185,E0034251,ADV,Strategy,AMER,L3,2023-09-07
1586,E0039658,ADV,People Advisory,APAC,L1,2023-04-25


In [49]:
# Because dates are sorted oldest to newest, keep="last"
# retains the most recent record for each employee.

hr_latest = hr_sorted.drop_duplicates(subset="Employee_ID",keep="last")

print("Rows before selecting latest records:", len(hr_sorted))
print("Rows after selecting latest records:", len(hr_latest))

Rows before selecting latest records: 2445
Rows after selecting latest records: 2400


Here, drop_duplicates() checks only Employee_ID because we specified subset. We are deliberately selecting one record per employee—not claiming their historical records were mistakes.

In [58]:
# Read the required division from the configuration.
required_division = config["scope"]["division"]

# Filter only AFTER selecting each employee's latest record.
hr_advisory = hr_latest[hr_latest["Division"] == required_division].copy()

print("Eligible Advisory employees:", len(hr_advisory))

hr_advisory.head()

Eligible Advisory employees: 1507


,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
2150,E0005667,ADV,Technology Consulting,APAC,L3,2025-05-07
2334,E0006083,ADV,People Advisory,APAC,L3,2024-03-20
2064,E0006249,ADV,People Advisory,EMEA,L2,2024-07-02
1147,E0017798,ADV,Risk,APAC,L1,2024-03-12
421,E0024994,ADV,Deals,AMER,L1,2024-01-07


question - Why this order? If someone moved from ADV to TAX, their latest TAX record must determine eligibility. Filtering to ADV first would wrongly retain their old record.

In [61]:
# Each employee should appear only once.
print("Repeated employee IDs:", hr_advisory["Employee_ID"].duplicated().sum())

# Every remaining record should belong to ADV.
print("Remaining divisions:", hr_advisory["Division"].unique())

Repeated employee IDs: 0
Remaining divisions: ['ADV']


In [63]:
hr_advisory["Division"].value_counts()

Division
ADV    1507
Name: count, dtype: int64

## Quill usage data
Explore the event log and prepare it for weekly user calculations.

In [66]:
# Load the Quill event log.
quill = pd.read_csv(data_dir / "quill_events.csv")

quill.head()

,user_id,event_time_utc,event_type,client
0,E6616097,2026-08-02T00:02:08Z,generation,mobile
1,E3050267,2026-08-02T00:12:28Z,generation,desktop
2,E3050267,2026-08-02T00:12:28Z,generation,desktop
3,E7932530,2026-08-02T00:18:01Z,generation,web
4,E6113456,2026-08-02T00:19:41Z,generation,desktop


In [68]:
# Check the number of rows, columns and their data types.
print("Rows and columns:", quill.shape)

quill.info()

Rows and columns: (94558, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94558 entries, 0 to 94557
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   user_id         94558 non-null  object
 1   event_time_utc  94558 non-null  object
 2   event_type      94558 non-null  object
 3   client          94558 non-null  object
dtypes: object(4)
memory usage: 2.9+ MB


In [74]:
# Count how many records belong to each type of action.

quill["event_type"].value_counts()

event_type
generation    72857
open          14405
rating         7296
Name: count, dtype: int64

In [72]:
# Check for missing values in each column.

quill.isna().sum()

user_id           0
event_time_utc    0
event_type        0
client            0
dtype: int64

generation = counted towards Quill’s active and power-user thresholds

open & no-generation = not an active user


In [77]:
# Preserve the original data and work on a copy.

quill_clean = quill.copy()

In [79]:
quill_clean.head()

,user_id,event_time_utc,event_type,client
0,E6616097,2026-08-02T00:02:08Z,generation,mobile
1,E3050267,2026-08-02T00:12:28Z,generation,desktop
2,E3050267,2026-08-02T00:12:28Z,generation,desktop
3,E7932530,2026-08-02T00:18:01Z,generation,web
4,E6113456,2026-08-02T00:19:41Z,generation,desktop


In [81]:
# Standardise IDs so they match the format used in HR.

quill_clean["user_id"] = quill_clean["user_id"].str.strip().str.upper()

In [83]:
quill_clean["user_id"].head(2)

0    E6616097
1    E3050267
Name: user_id, dtype: object

In [85]:
# Count records where the ID formatting changed.

changed_ids = (quill["user_id"] != quill_clean["user_id"]).sum()

print("Rows with ID formatting changes:", changed_ids)

Rows with ID formatting changes: 8455


In [95]:
# Count exact duplicate rows in the original file.
print("Raw duplicate rows:", quill.duplicated().sum())

Raw duplicate rows: 1397


In [97]:
# Check again after standardising IDs.
print("Duplicate rows after ID cleaning:",quill_clean.duplicated().sum())

Duplicate rows after ID cleaning: 1397


In [103]:
# Display all occurrences of duplicate rows so we can inspect them.

duplicate_events = quill_clean[quill_clean.duplicated(keep=False)]

duplicate_events.sort_values(["user_id", "event_time_utc"]).head(10)

,user_id,event_time_utc,event_type,client
88759,E0014152,2026-09-30T21:26:53Z,generation,web
88760,E0014152,2026-09-30T21:26:53Z,generation,web
734,E0024994,2026-08-04T07:48:42Z,generation,mobile
735,E0024994,2026-08-04T07:48:42Z,generation,mobile
14228,E0024994,2026-08-17T02:49:36Z,generation,mobile
14229,E0024994,2026-08-17T02:49:36Z,generation,mobile
64821,E0024994,2026-09-17T11:28:26Z,generation,desktop
64822,E0024994,2026-09-17T11:28:26Z,generation,desktop
69696,E0049329,2026-09-21T00:07:24Z,generation,web
69697,E0049329,2026-09-21T00:07:24Z,generation,web


In [107]:
# Convert event timestamps from text to UTC datetime values.

quill_clean["event_time_utc"] = pd.to_datetime(quill_clean["event_time_utc"],format="%Y-%m-%dT%H:%M:%SZ",utc=True,errors="raise")

# Check the converted column type and sample values.
print(quill_clean["event_time_utc"].dtype)
quill_clean["event_time_utc"].head()

datetime64[ns, UTC]


0   2026-08-02 00:02:08+00:00
1   2026-08-02 00:12:28+00:00
2   2026-08-02 00:12:28+00:00
3   2026-08-02 00:18:01+00:00
4   2026-08-02 00:19:41+00:00
Name: event_time_utc, dtype: datetime64[ns, UTC]

In [115]:
# Calculate the number of days since Sunday for each event.

days_since_sunday = (quill_clean["event_time_utc"].dt.dayofweek + 1) % 7

# Display the first five results.
days_since_sunday

0        0
1        0
2        0
3        0
4        0
        ..
94553    6
94554    6
94555    6
94556    6
94557    6
Name: event_time_utc, Length: 94558, dtype: int32

In [117]:
# Assign each event to its Sunday week start at midnight UTC.
quill_clean["week_start"] = (quill_clean["event_time_utc"].dt.normalize() - pd.to_timedelta(days_since_sunday, unit="D"))

# Compare event timestamps with their reporting week starts.
quill_clean[["event_time_utc", "week_start"]].head(10)

,event_time_utc,week_start
0,2026-08-02 00:02:08+00:00,2026-08-02 00:00:00+00:00
1,2026-08-02 00:12:28+00:00,2026-08-02 00:00:00+00:00
2,2026-08-02 00:12:28+00:00,2026-08-02 00:00:00+00:00
3,2026-08-02 00:18:01+00:00,2026-08-02 00:00:00+00:00
4,2026-08-02 00:19:41+00:00,2026-08-02 00:00:00+00:00
5,2026-08-02 00:19:55+00:00,2026-08-02 00:00:00+00:00
6,2026-08-02 00:23:58+00:00,2026-08-02 00:00:00+00:00
7,2026-08-02 00:24:47+00:00,2026-08-02 00:00:00+00:00
8,2026-08-02 00:26:19+00:00,2026-08-02 00:00:00+00:00
9,2026-08-02 00:36:32+00:00,2026-08-02 00:00:00+00:00


In [121]:
# Count available event rows for each reporting week.
quill_weekly_coverage = quill_clean.groupby("week_start").size()

# Display weekly event counts in date order.
quill_weekly_coverage.sort_index()

week_start
2026-08-02 00:00:00+00:00     2232
2026-08-09 00:00:00+00:00    10284
2026-08-16 00:00:00+00:00    10670
2026-08-23 00:00:00+00:00    11889
2026-08-30 00:00:00+00:00    12261
2026-09-06 00:00:00+00:00    12236
2026-09-13 00:00:00+00:00     8191
2026-09-20 00:00:00+00:00    13660
2026-09-27 00:00:00+00:00    13135
dtype: int64

**Reporting period**

In [127]:
# Read the reporting boundaries from the configuration as UTC datetimes.
first_week = pd.to_datetime(config["reporting_window"]["first_week_start"], utc=True)

last_week = pd.to_datetime(config["reporting_window"]["last_week_start"], utc=True)

# Keep events belonging to the requested reporting weeks.
quill_window = quill_clean[(quill_clean["week_start"] >= first_week) & (quill_clean["week_start"] <= last_week)].copy()

# Check the remaining weeks and their event counts.
quill_window.groupby("week_start").size()

week_start
2026-08-09 00:00:00+00:00    10284
2026-08-16 00:00:00+00:00    10670
2026-08-23 00:00:00+00:00    11889
2026-08-30 00:00:00+00:00    12261
2026-09-06 00:00:00+00:00    12236
2026-09-13 00:00:00+00:00     8191
2026-09-20 00:00:00+00:00    13660
2026-09-27 00:00:00+00:00    13135
dtype: int64

In [132]:
# Keep events belonging to employees in the latest Advisory roster.
quill_advisory = quill_window[quill_window["user_id"].isin(hr_advisory["Employee_ID"])].copy()

# Check the number of events and distinct users retained.
print("Advisory event rows:", len(quill_advisory))
print("Distinct Advisory users:", quill_advisory["user_id"].nunique())

Advisory event rows: 58225
Distinct Advisory users: 1093


In [136]:
# Keep generation events used to calculate Quill activity thresholds.
quill_generations = quill_advisory[quill_advisory["event_type"] == "generation"].copy()

# Check the retained event type and number of rows.
print(quill_generations["event_type"].value_counts())

event_type
generation    44857
Name: count, dtype: int64


In [138]:
# Count generation event rows for each employee in each week.
# Duplicate records are still included at this stage.
quill_user_week = (quill_generations.groupby(["week_start", "user_id"]).size().reset_index(name="generation_count"))

# Display the first ten employee-week totals.
quill_user_week.head(10)

,week_start,user_id,generation_count
0,2026-08-09 00:00:00+00:00,E0024994,18
1,2026-08-09 00:00:00+00:00,E0049329,8
2,2026-08-09 00:00:00+00:00,E0099913,8
3,2026-08-09 00:00:00+00:00,E0126069,19
4,2026-08-09 00:00:00+00:00,E0198256,9
5,2026-08-09 00:00:00+00:00,E0201638,13
6,2026-08-09 00:00:00+00:00,E0212609,12
7,2026-08-09 00:00:00+00:00,E0227635,5
8,2026-08-09 00:00:00+00:00,E0291431,11
9,2026-08-09 00:00:00+00:00,E0294001,8


In [144]:
quill_user_week.tail(10)

,week_start,user_id,generation_count
4253,2026-09-27 00:00:00+00:00,E9713960,12
4254,2026-09-27 00:00:00+00:00,E9744321,16
4255,2026-09-27 00:00:00+00:00,E9753451,5
4256,2026-09-27 00:00:00+00:00,E9796750,10
4257,2026-09-27 00:00:00+00:00,E9824303,9
4258,2026-09-27 00:00:00+00:00,E9833834,6
4259,2026-09-27 00:00:00+00:00,E9896303,12
4260,2026-09-27 00:00:00+00:00,E9940596,2
4261,2026-09-27 00:00:00+00:00,E9972598,7
4262,2026-09-27 00:00:00+00:00,E9988141,9


In [148]:
# Create a comparison dataset with one copy of each identical event.

quill_generations_dedup = quill_generations.drop_duplicates(subset=["user_id", "event_time_utc", "event_type", "client"], keep="first")

quill_generations_dedup

,user_id,event_time_utc,event_type,client,week_start
2232,E7132004,2026-08-09 00:01:58+00:00,generation,mobile,2026-08-09 00:00:00+00:00
2233,E4826281,2026-08-09 00:02:37+00:00,generation,desktop,2026-08-09 00:00:00+00:00
2235,E4543194,2026-08-09 00:02:53+00:00,generation,web,2026-08-09 00:00:00+00:00
2236,E8018160,2026-08-09 00:03:02+00:00,generation,desktop,2026-08-09 00:00:00+00:00
2241,E6443127,2026-08-09 00:06:32+00:00,generation,web,2026-08-09 00:00:00+00:00
...,...,...,...,...,...
94536,E1535077,2026-10-03 23:47:12+00:00,generation,desktop,2026-09-27 00:00:00+00:00
94542,E6350694,2026-10-03 23:52:23+00:00,generation,mobile,2026-09-27 00:00:00+00:00
94543,E9340705,2026-10-03 23:52:39+00:00,generation,desktop,2026-09-27 00:00:00+00:00
94551,E8299558,2026-10-03 23:56:16+00:00,generation,desktop,2026-09-27 00:00:00+00:00


In [154]:
# Count weekly generations after removing identical event copies.

quill_user_week_dedup = (quill_generations_dedup.groupby(["week_start", "user_id"]).size().reset_index(name="generation_count_dedup"))
quill_user_week_dedup.head(10)

,week_start,user_id,generation_count_dedup
0,2026-08-09 00:00:00+00:00,E0024994,18
1,2026-08-09 00:00:00+00:00,E0049329,8
2,2026-08-09 00:00:00+00:00,E0099913,8
3,2026-08-09 00:00:00+00:00,E0126069,19
4,2026-08-09 00:00:00+00:00,E0198256,9
5,2026-08-09 00:00:00+00:00,E0201638,12
6,2026-08-09 00:00:00+00:00,E0212609,12
7,2026-08-09 00:00:00+00:00,E0227635,5
8,2026-08-09 00:00:00+00:00,E0291431,11
9,2026-08-09 00:00:00+00:00,E0294001,8


In [156]:
# Compare counts with and without identical event copies.

quill_duplicate_comparison = quill_user_week.merge(quill_user_week_dedup,on=["week_start", "user_id"],how="left",validate="one_to_one")
quill_duplicate_comparison

,week_start,user_id,generation_count,generation_count_dedup
0,2026-08-09 00:00:00+00:00,E0024994,18,18
1,2026-08-09 00:00:00+00:00,E0049329,8,8
2,2026-08-09 00:00:00+00:00,E0099913,8,8
3,2026-08-09 00:00:00+00:00,E0126069,19,19
4,2026-08-09 00:00:00+00:00,E0198256,9,9
...,...,...,...,...
4258,2026-09-27 00:00:00+00:00,E9833834,6,6
4259,2026-09-27 00:00:00+00:00,E9896303,12,12
4260,2026-09-27 00:00:00+00:00,E9940596,2,2
4261,2026-09-27 00:00:00+00:00,E9972598,7,7


In [158]:
# Display employee-weeks where the generation count changed.

quill_duplicate_comparison[quill_duplicate_comparison["generation_count"]!= quill_duplicate_comparison["generation_count_dedup"]].head(10)

,week_start,user_id,generation_count,generation_count_dedup
5,2026-08-09 00:00:00+00:00,E0201638,13,12
19,2026-08-09 00:00:00+00:00,E0426839,9,8
41,2026-08-09 00:00:00+00:00,E0971255,17,16
47,2026-08-09 00:00:00+00:00,E1209178,6,5
49,2026-08-09 00:00:00+00:00,E1275842,13,12
56,2026-08-09 00:00:00+00:00,E1377170,13,12
68,2026-08-09 00:00:00+00:00,E1596518,9,8
100,2026-08-09 00:00:00+00:00,E2375422,8,7
110,2026-08-09 00:00:00+00:00,E2512942,16,14
120,2026-08-09 00:00:00+00:00,E2659090,11,9


13 → 12: dono cases mein power user.
16 → 14: dono cases mein power user.
11 → 9: dono cases mein power user nahi.
12 → 11 hota: tab classification change hoti—pehle power user, baad mein nahi.

In [164]:
# Read the Quill power-user threshold from the configuration.
for tool in config["tools"]:
    if tool["tool_id"] == "quill":
        quill_power_threshold = tool["wpu_threshold"]

# Identify employee-weeks that fall below the threshold after deduplication.
power_status_changed = quill_duplicate_comparison[(quill_duplicate_comparison["generation_count"] >= quill_power_threshold)
    & (quill_duplicate_comparison["generation_count_dedup"]< quill_power_threshold ) ].copy()

# Show the threshold and the number of affected employee-weeks.
print("Power-user threshold:", quill_power_threshold)
print("Employee-weeks with changed power status:", len(power_status_changed))

power_status_changed.head(10)

Power-user threshold: 12
Employee-weeks with changed power status: 64


,week_start,user_id,generation_count,generation_count_dedup
387,2026-08-09 00:00:00+00:00,E7945229,13,11
507,2026-08-16 00:00:00+00:00,E0396272,12,11
532,2026-08-16 00:00:00+00:00,E1024004,12,11
591,2026-08-16 00:00:00+00:00,E2297758,12,11
737,2026-08-16 00:00:00+00:00,E5287818,12,11
795,2026-08-16 00:00:00+00:00,E6350694,12,11
962,2026-08-16 00:00:00+00:00,E9358352,12,11
1055,2026-08-23 00:00:00+00:00,E1292611,12,11
1073,2026-08-23 00:00:00+00:00,E1529167,12,11
1239,2026-08-23 00:00:00+00:00,E4826281,12,11


In [166]:
# Count changed power-user classifications in each affected week.
weekly_power_impact = power_status_changed.groupby("week_start").size()

weekly_power_impact

week_start
2026-08-09 00:00:00+00:00     1
2026-08-16 00:00:00+00:00     6
2026-08-23 00:00:00+00:00     9
2026-08-30 00:00:00+00:00     6
2026-09-06 00:00:00+00:00    10
2026-09-13 00:00:00+00:00     8
2026-09-20 00:00:00+00:00    15
2026-09-27 00:00:00+00:00     9
dtype: int64

In [170]:
#Use deduplicated generation counts for the reporting calculation.
quill_user_week_final = quill_user_week_dedup.copy()

# Read the Quill active-user threshold from the configuration.
for tool in config["tools"]:
    if tool["tool_id"] == "quill":
        quill_active_threshold = tool["wau_threshold"]

# Flag employee-weeks that meet each usage threshold.
quill_user_week_final["is_active"] = (quill_user_week_final["generation_count_dedup"] >= quill_active_threshold)

quill_user_week_final["is_power"] = (quill_user_week_final["generation_count_dedup"] >= quill_power_threshold)

quill_user_week_final.head(10)

,week_start,user_id,generation_count_dedup,is_active,is_power
0,2026-08-09 00:00:00+00:00,E0024994,18,True,True
1,2026-08-09 00:00:00+00:00,E0049329,8,True,False
2,2026-08-09 00:00:00+00:00,E0099913,8,True,False
3,2026-08-09 00:00:00+00:00,E0126069,19,True,True
4,2026-08-09 00:00:00+00:00,E0198256,9,True,False
5,2026-08-09 00:00:00+00:00,E0201638,12,True,True
6,2026-08-09 00:00:00+00:00,E0212609,12,True,True
7,2026-08-09 00:00:00+00:00,E0227635,5,True,False
8,2026-08-09 00:00:00+00:00,E0291431,11,True,False
9,2026-08-09 00:00:00+00:00,E0294001,8,True,False


In [172]:
# Count active and power users in each week by summing Boolean flags.
quill_weekly = quill_user_week_final.groupby("week_start")[["is_active", "is_power"]].sum()

# Rename the totals to the required reporting metric names.
quill_weekly = quill_weekly.rename(columns={"is_active": "wau", "is_power": "wpu"}).reset_index()

quill_weekly

,week_start,wau,wpu
0,2026-08-09 00:00:00+00:00,490,187
1,2026-08-16 00:00:00+00:00,501,196
2,2026-08-23 00:00:00+00:00,535,206
3,2026-08-30 00:00:00+00:00,560,213
4,2026-09-06 00:00:00+00:00,550,202
5,2026-09-13 00:00:00+00:00,381,135
6,2026-09-20 00:00:00+00:00,641,231
7,2026-09-27 00:00:00+00:00,605,222


In [176]:
# Calculate the proportion of active users who are power users.
# Leave the rate empty when WAU is zero to avoid division by zero.
quill_weekly["wpu_rate"] = (quill_weekly["wpu"]/ quill_weekly["wau"].where(quill_weekly["wau"] > 0)).round(4)

quill_weekly

,week_start,wau,wpu,wpu_rate
0,2026-08-09 00:00:00+00:00,490,187,0.3816
1,2026-08-16 00:00:00+00:00,501,196,0.3912
2,2026-08-23 00:00:00+00:00,535,206,0.3850
3,2026-08-30 00:00:00+00:00,560,213,0.3804
4,2026-09-06 00:00:00+00:00,550,202,0.3673
5,2026-09-13 00:00:00+00:00,381,135,0.3543
6,2026-09-20 00:00:00+00:00,641,231,0.3604
7,2026-09-27 00:00:00+00:00,605,222,0.3669


In [178]:
# Identify the tool represented by these metrics.
quill_weekly["tool_id"] = "quill"

# Mark weeks that have source events before business filtering.
quill_weekly["measured"] = quill_weekly["week_start"].isin(quill_weekly_coverage.index)

# Arrange columns in the order required by the brief.
quill_weekly = quill_weekly[["week_start", "tool_id", "wau", "wpu", "wpu_rate", "measured"]]

quill_weekly

,week_start,tool_id,wau,wpu,wpu_rate,measured
0,2026-08-09 00:00:00+00:00,quill,490,187,0.3816,True
1,2026-08-16 00:00:00+00:00,quill,501,196,0.3912,True
2,2026-08-23 00:00:00+00:00,quill,535,206,0.3850,True
3,2026-08-30 00:00:00+00:00,quill,560,213,0.3804,True
4,2026-09-06 00:00:00+00:00,quill,550,202,0.3673,True
5,2026-09-13 00:00:00+00:00,quill,381,135,0.3543,True
6,2026-09-20 00:00:00+00:00,quill,641,231,0.3604,True
7,2026-09-27 00:00:00+00:00,quill,605,222,0.3669,True


# ATLAS EDA

In [181]:
# Inspect the first Atlas delivery without assuming a header row.
atlas_preview = pd.read_excel(data_dir / "atlas" / "atlas_usage_20260817.xlsx",sheet_name="Usage",header=None)

# Display the first eight rows to inspect the report layout.
atlas_preview.head(8)

,0,1,2,3
0,Atlas - Licence Usage Report (extracted 2026-0...,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,User ID,Week Ending,Queries Run,Active Days
3,E0024994,2026-08-15 00:00:00,8,1
4,E0028755,2026-08-15 00:00:00,21,4
5,E0034251,2026-08-15 00:00:00,14,3
6,E0050471,2026-08-15 00:00:00,0,0
7,E0088184,2026-08-15 00:00:00,25,5


In [183]:
# Load the first Atlas delivery using the third row as column names.

atlas_first = pd.read_excel(data_dir / "atlas" / "atlas_usage_20260817.xlsx",sheet_name="Usage",header=2)

# Display the first five data rows.

atlas_first.head()

,User ID,Week Ending,Queries Run,Active Days
0,E0024994,2026-08-15,8,1
1,E0028755,2026-08-15,21,4
2,E0034251,2026-08-15,14,3
3,E0050471,2026-08-15,0,0
4,E0088184,2026-08-15,25,5


In [185]:
# Inspect column names, non-null counts, and data types.

atlas_first.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   User ID      506 non-null    object        
 1   Week Ending  506 non-null    datetime64[ns]
 2   Queries Run  506 non-null    int64         
 3   Active Days  506 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(1)
memory usage: 15.9+ KB


In [187]:
# Find and sort all Atlas usage Excel files.
atlas_files = sorted((data_dir / "atlas").glob("atlas_usage_*.xlsx"))

# Display the number of files and their names.
print("Atlas files found:", len(atlas_files))

for file in atlas_files:
    print(file.name)

Atlas files found: 8
atlas_usage_20260817.xlsx
atlas_usage_20260824.xlsx
atlas_usage_20260831.xlsx
atlas_usage_20260907.xlsx
atlas_usage_20260913.xlsx
atlas_usage_20260914.xlsx
atlas_usage_20260921.xlsx
atlas_usage_20261005.xlsx


In [189]:
# Store each Atlas delivery as a separate DataFrame.
atlas_tables = []

for file in atlas_files:
    # Read the usage data using the third row as column names.
    atlas_table = pd.read_excel(file, sheet_name="Usage", header=2)

    # Preserve the source filename for tracing each record.
    atlas_table["source_file"] = file.name

    # Add this delivery to the list.
    atlas_tables.append(atlas_table)

# Display a sample from the first delivery.
atlas_tables[0].head()

,User ID,Week Ending,Queries Run,Active Days,source_file
0,E0024994,2026-08-15,8,1,atlas_usage_20260817.xlsx
1,E0028755,2026-08-15,21,4,atlas_usage_20260817.xlsx
2,E0034251,2026-08-15,14,3,atlas_usage_20260817.xlsx
3,E0050471,2026-08-15,0,0,atlas_usage_20260817.xlsx
4,E0088184,2026-08-15,25,5,atlas_usage_20260817.xlsx


In [191]:
# Combine all Atlas deliveries into one DataFrame.
atlas_raw = pd.concat(atlas_tables, ignore_index=True)

# Display the total number of rows and a sample.
print("Total Atlas rows:", len(atlas_raw))
atlas_raw.head()

Total Atlas rows: 4682


,User ID,Week Ending,Queries Run,Active Days,source_file
0,E0024994,2026-08-15,8,1,atlas_usage_20260817.xlsx
1,E0028755,2026-08-15,21,4,atlas_usage_20260817.xlsx
2,E0034251,2026-08-15,14,3,atlas_usage_20260817.xlsx
3,E0050471,2026-08-15,0,0,atlas_usage_20260817.xlsx
4,E0088184,2026-08-15,25,5,atlas_usage_20260817.xlsx


In [195]:
# Count rows by source delivery and reporting week.
atlas_delivery_summary = (atlas_raw.groupby(["source_file", "Week Ending"]).size().reset_index(name="row_count"))

atlas_delivery_summary

,source_file,Week Ending,row_count
0,atlas_usage_20260817.xlsx,2026-08-15,506
1,atlas_usage_20260824.xlsx,2026-08-22,537
2,atlas_usage_20260831.xlsx,2026-08-29,565
3,atlas_usage_20260907.xlsx,2026-09-05,518
4,atlas_usage_20260913.xlsx,2026-09-05,602
5,atlas_usage_20260914.xlsx,2026-09-12,629
6,atlas_usage_20260921.xlsx,2026-09-19,652
7,atlas_usage_20261005.xlsx,2026-10-03,673


In [197]:
# Preserve the combined source data before cleaning.
atlas_clean = atlas_raw.copy()

# Extract the delivery date from each source filename.
atlas_clean["delivery_date"] = pd.to_datetime(atlas_clean["source_file"].str.replace("atlas_usage_", "", regex=False).str.replace(".xlsx", "", regex=False),format="%Y%m%d")

# Display each source file and its extracted delivery date once.
atlas_clean[["source_file", "delivery_date"]].drop_duplicates()

,source_file,delivery_date
0,atlas_usage_20260817.xlsx,2026-08-17
506,atlas_usage_20260824.xlsx,2026-08-24
1043,atlas_usage_20260831.xlsx,2026-08-31
1608,atlas_usage_20260907.xlsx,2026-09-07
2126,atlas_usage_20260913.xlsx,2026-09-13
2728,atlas_usage_20260914.xlsx,2026-09-14
3357,atlas_usage_20260921.xlsx,2026-09-21
4009,atlas_usage_20261005.xlsx,2026-10-05


In [199]:
# Find the latest delivery date for each reporting week.

atlas_clean["latest_delivery_date"] = (atlas_clean.groupby("Week Ending")["delivery_date"].transform("max"))

# Inspect the delivery dates against the latest date for each week.

atlas_clean[["Week Ending", "source_file", "delivery_date", "latest_delivery_date"]].drop_duplicates()

,Week Ending,source_file,delivery_date,latest_delivery_date
0,2026-08-15,atlas_usage_20260817.xlsx,2026-08-17,2026-08-17
506,2026-08-22,atlas_usage_20260824.xlsx,2026-08-24,2026-08-24
1043,2026-08-29,atlas_usage_20260831.xlsx,2026-08-31,2026-08-31
1608,2026-09-05,atlas_usage_20260907.xlsx,2026-09-07,2026-09-13
2126,2026-09-05,atlas_usage_20260913.xlsx,2026-09-13,2026-09-13
2728,2026-09-12,atlas_usage_20260914.xlsx,2026-09-14,2026-09-14
3357,2026-09-19,atlas_usage_20260921.xlsx,2026-09-21,2026-09-21
4009,2026-10-03,atlas_usage_20261005.xlsx,2026-10-05,2026-10-05


In [205]:
# Retain only the latest delivery for each reporting week.

atlas_latest = atlas_clean[atlas_clean["delivery_date"] == atlas_clean["latest_delivery_date"]].copy()

# Check how many rows were retained and excluded.
print("Rows retained:", len(atlas_latest))
print("Superseded rows excluded:", len(atlas_clean) - len(atlas_latest))


Rows retained: 4164
Superseded rows excluded: 518


In [207]:
# Standardise employee IDs for consistent matching with the HR roster.

atlas_latest["User ID"] = (atlas_latest["User ID"].str.strip().str.upper())

# Display the first five standardised IDs.
atlas_latest["User ID"].head()

0    E0024994
1    E0028755
2    E0034251
3    E0050471
4    E0088184
Name: User ID, dtype: object

In [209]:
# Inspect all repeated employee-week records in the latest deliveries.

atlas_repeated_users = atlas_latest[atlas_latest.duplicated(subset=["User ID", "Week Ending"], keep=False)]

print("Rows with repeated employee-week combinations:", len(atlas_repeated_users))
atlas_repeated_users.head(10)

Rows with repeated employee-week combinations: 0


,User ID,Week Ending,Queries Run,Active Days,source_file,delivery_date,latest_delivery_date


In [211]:
# Count missing values in the fields required for Atlas metrics.

atlas_latest[["User ID", "Week Ending", "Queries Run", "Active Days"]].isna().sum()

User ID        0
Week Ending    0
Queries Run    0
Active Days    0
dtype: int64

In [213]:
# Inspect the distribution and range of the usage measures.
atlas_latest[["Queries Run", "Active Days"]].describe()

,Queries Run,Active Days
count,4164.000000,4164.000000
mean,11.432037,1.964697
std,11.135505,1.883453
min,0.000000,0.000000
25%,0.000000,0.000000
50%,11.000000,2.000000
75%,21.000000,4.000000
max,45.000000,5.000000


In [215]:
# Inspect records with queries recorded but no active days.

atlas_usage_conflicts = atlas_latest[(atlas_latest["Queries Run"] > 0)& (atlas_latest["Active Days"] == 0)]

print("Rows with queries but no active days:", len(atlas_usage_conflicts))
atlas_usage_conflicts.head(10)

Rows with queries but no active days: 0


,User ID,Week Ending,Queries Run,Active Days,source_file,delivery_date,latest_delivery_date


In [217]:
# Check which weekdays appear in the vendor's week-ending dates.
atlas_latest["Week Ending"].dt.day_name().value_counts()

Week Ending
Saturday    4164
Name: count, dtype: int64

In [219]:
# Convert Saturday week-ending dates to Sunday week starts in UTC.

atlas_latest["week_start"] = (pd.to_datetime(atlas_latest["Week Ending"], utc=True).dt.normalize()- pd.Timedelta(days=6))

# Display the distinct week-ending and week-start pairs.
atlas_latest[["Week Ending", "week_start"]].drop_duplicates()

,Week Ending,week_start
0,2026-08-15,2026-08-09 00:00:00+00:00
506,2026-08-22,2026-08-16 00:00:00+00:00
1043,2026-08-29,2026-08-23 00:00:00+00:00
2126,2026-09-05,2026-08-30 00:00:00+00:00
2728,2026-09-12,2026-09-06 00:00:00+00:00
3357,2026-09-19,2026-09-13 00:00:00+00:00
4009,2026-10-03,2026-09-27 00:00:00+00:00


In [221]:
# Retain Atlas records within the configured reporting window.
atlas_window = atlas_latest[(atlas_latest["week_start"] >= first_week)& (atlas_latest["week_start"] <= last_week)].copy()

# Record source coverage before filtering to Advisory employees.
atlas_weekly_coverage = atlas_window.groupby("week_start").size()

atlas_weekly_coverage

week_start
2026-08-09 00:00:00+00:00    506
2026-08-16 00:00:00+00:00    537
2026-08-23 00:00:00+00:00    565
2026-08-30 00:00:00+00:00    602
2026-09-06 00:00:00+00:00    629
2026-09-13 00:00:00+00:00    652
2026-09-27 00:00:00+00:00    673
dtype: int64

In [223]:
# Retain employees in the latest eligible Advisory roster.

atlas_advisory = atlas_window[atlas_window["User ID"].isin(hr_advisory["Employee_ID"])].copy()

# Display the retained row count and a sample.
print("Atlas Advisory employee-week rows:", len(atlas_advisory))
atlas_advisory.head()

Atlas Advisory employee-week rows: 3154


,User ID,Week Ending,Queries Run,Active Days,source_file,delivery_date,latest_delivery_date,week_start
0,E0024994,2026-08-15,8,1,atlas_usage_20260817.xlsx,2026-08-17,2026-08-17,2026-08-09 00:00:00+00:00
1,E0028755,2026-08-15,21,4,atlas_usage_20260817.xlsx,2026-08-17,2026-08-17,2026-08-09 00:00:00+00:00
2,E0034251,2026-08-15,14,3,atlas_usage_20260817.xlsx,2026-08-17,2026-08-17,2026-08-09 00:00:00+00:00
4,E0088184,2026-08-15,25,5,atlas_usage_20260817.xlsx,2026-08-17,2026-08-17,2026-08-09 00:00:00+00:00
5,E0099913,2026-08-15,19,4,atlas_usage_20260817.xlsx,2026-08-17,2026-08-17,2026-08-09 00:00:00+00:00


In [225]:
# Read the Atlas user thresholds from the configuration.
for tool in config["tools"]:
    if tool["tool_id"] == "atlas":
        atlas_active_threshold = tool["wau_threshold"]
        atlas_power_threshold = tool["wpu_threshold"]

# Display the thresholds used to classify Atlas users.
print("Active-user threshold (active days):", atlas_active_threshold)
print("Power-user threshold (queries run):", atlas_power_threshold)

Active-user threshold (active days): 1
Power-user threshold (queries run): 25


In [227]:
# Flag employee-weeks that meet the Atlas activity threshold.
atlas_advisory["is_active"] = (atlas_advisory["Active Days"] >= atlas_active_threshold)

# Flag employee-weeks that meet the Atlas power-user threshold.
atlas_advisory["is_power"] = (atlas_advisory["Queries Run"] >= atlas_power_threshold)

# Display the usage measures and classification flags.
atlas_advisory[
    ["week_start", "User ID", "Queries Run", "Active Days", "is_active", "is_power"]
].head(10)

,week_start,User ID,Queries Run,Active Days,is_active,is_power
0,2026-08-09 00:00:00+00:00,E0024994,8,1,True,False
1,2026-08-09 00:00:00+00:00,E0028755,21,4,True,False
2,2026-08-09 00:00:00+00:00,E0034251,14,3,True,False
4,2026-08-09 00:00:00+00:00,E0088184,25,5,True,True
5,2026-08-09 00:00:00+00:00,E0099913,19,4,True,False
6,2026-08-09 00:00:00+00:00,E0112089,23,5,True,False
8,2026-08-09 00:00:00+00:00,E0135426,0,0,False,False
10,2026-08-09 00:00:00+00:00,E0164205,25,5,True,True
11,2026-08-09 00:00:00+00:00,E0181954,0,0,False,False
13,2026-08-09 00:00:00+00:00,E0190949,0,0,False,False


In [229]:
# Count active and power users for each reporting week.

atlas_weekly = (atlas_advisory.groupby("week_start")[["is_active", "is_power"]].sum().rename(columns={"is_active": "wau", "is_power": "wpu"}).reset_index())

atlas_weekly

,week_start,wau,wpu
0,2026-08-09 00:00:00+00:00,235,65
1,2026-08-16 00:00:00+00:00,229,60
2,2026-08-23 00:00:00+00:00,247,66
3,2026-08-30 00:00:00+00:00,272,77
4,2026-09-06 00:00:00+00:00,282,67
5,2026-09-13 00:00:00+00:00,277,74
6,2026-09-27 00:00:00+00:00,312,84
